# PPS Experiment – XDF extractor

This notebook loads a LabRecorder **.xdf** file, finds the **Audio** and **MouseEvents** streams produced by the PPS auto‑streamer, and saves them as:

* `...Audio....wav` – float32 WAV at the stream's native sample‑rate
* `...MouseClicks....csv` – a CSV with columns `rel_time_sec, x_px, y_px`

> **Tip:** if you haven't installed the required libraries in this environment yet, run the first code cell.

In [1]:
!pip install pyxdf numpy soundfile pandas --quiet
import os, pathlib, re, json
from typing import Optional

import numpy as np
import pandas as pd
import soundfile as sf
import pyxdf


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\cogpsy-v

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\cogpsy-v

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



In [2]:
def find_stream(streams, *, type_=None, name_regex=None, channel_count=None):
    """Return the first stream that matches all given criteria."""
    for s in streams:
        info = s['info']
        name = info['name'][0]
        stype = info['type'][0]
        nch = int(info['channel_count'][0])
        if type_ is not None and stype != type_:
            continue
        if channel_count is not None and nch != channel_count:
            continue
        if name_regex is not None and re.fullmatch(name_regex, name) is None:
            continue
        return s
    return None

In [3]:
# ---- user inputs -------------------------------------------------------
xdf_path = r"C:\Users\cogpsy-vrlab\Desktop\XDF\sub-P001_task-PPS_recording-%t_desc-%d.xdf"
out_dir  = pathlib.Path(r"C:\Users\cogpsy-vrlab\Desktop\XDF")   # <─ fixed output folder
# -----------------------------------------------------------------------

out_dir.mkdir(parents=True, exist_ok=True)

print(f"Loading {xdf_path} ...")
streams, _ = pyxdf.load_xdf(xdf_path)
print(f"Loaded {len(streams)} streams")

# Identify audio stream (2 channels, type 'Audio')
audio_stream = (
    find_stream(streams, type_="Audio", channel_count=2)
    or find_stream(streams, name_regex=r".*_Audio_.*")
)

# Identify mouse‑click stream (3 channels, type 'MouseEvents')
mouse_stream = (
    find_stream(streams, type_="MouseEvents", channel_count=3)
    or find_stream(streams, name_regex=r".*_MouseClicks_.*")
)

print("\n--- streams detected ---")
print("Audio:", audio_stream['info']['name'][0] if audio_stream else "not found")
print("Mouse:", mouse_stream['info']['name'][0] if mouse_stream else "not found")

Loading C:\Users\cogpsy-vrlab\Desktop\XDF\sub-P001_task-PPS_recording-%t_desc-%d.xdf ...
Loaded 2 streams

--- streams detected ---
Audio: PPS_Unknown_Audio_20250429_180700
Mouse: PPS_Unknown_MouseClicks_20250429_180700


In [4]:
# ---- save audio --------------------------------------------------------
if audio_stream:
    name = audio_stream["info"]["name"][0]
    sr   = int(float(audio_stream["info"]["nominal_srate"][0]))   # e.g. "44100.0000"

    ts = audio_stream["time_series"]           # alias for readability
    if len(ts) == 0:                           # <─ robust emptiness check
        print(f"⚠️  Stream '{name}' contains 0 samples – nothing to write.")
    else:
        samples = np.vstack(ts).astype(np.float32).clip(-1.0, 1.0)
        wav_path = out_dir / f"{name}.wav"
        sf.write(wav_path, samples, sr)
        print(f"✓ Saved audio -> {wav_path}   shape={samples.shape}, sr={sr}")
else:
    print("No audio stream found")


✓ Saved audio -> C:\Users\cogpsy-vrlab\Desktop\XDF\PPS_Unknown_Audio_20250429_180700.wav   shape=(2228224, 2), sr=44100


In [5]:
# ---- save mouse clicks -------------------------------------------------
if mouse_stream:
    name = mouse_stream['info']['name'][0]
    click_data = np.array(mouse_stream['time_series'], dtype=np.float64)
    df_clicks = pd.DataFrame(click_data, columns=['rel_time_sec', 'x_px', 'y_px'])
    csv_path = out_dir / f"{name}.csv"
    df_clicks.to_csv(csv_path, index=False)
    print(f"Saved mouse clicks -> {csv_path}  rows={len(df_clicks)}")
else:
    print("No mouse stream to save")

Saved mouse clicks -> C:\Users\cogpsy-vrlab\Desktop\XDF\PPS_Unknown_MouseClicks_20250429_180700.csv  rows=99


In [6]:
# --- perfectly aligned audio + click plot ------------------------------------
import pathlib, numpy as np, pandas as pd, pyxdf, matplotlib.pyplot as plt

# ⇩  Point to the XDF you just recorded  (here: newest file in the folder)
xdf_dir  = pathlib.Path(r"C:\Users\cogpsy-vrlab\Desktop\XDF")
xdf_path = max(xdf_dir.glob("*.xdf"), key=lambda p: p.stat().st_mtime)

print("Loading", xdf_path.name)
streams, _ = pyxdf.load_xdf(xdf_path)

# ---- pick the audio + mouse streams ----
audio = next(s for s in streams if s["info"]["type"][0] == "Audio")
mouse = next(s for s in streams if s["info"]["type"][0] == "MouseEvents")

# audio data + timestamps
aud_data  = np.vstack(audio["time_series"]).astype(np.float32)       # (N, ch)
aud_ts    = np.asarray(audio["time_stamps"])                         # (N,)
aud_mono  = aud_data.mean(axis=1)                                    # mono
t0        = aud_ts[0]                                                # common zero

# mouse click timestamps (already relative to stream start in your code,
# but we read the absolute LSL ts here to guarantee sync)
click_ts  = np.asarray(mouse["time_stamps"]) - t0

# ---- plot ----
plt.figure(figsize=(14,4))
plt.plot(aud_ts - t0, aud_mono, label="audio (mono)")
for t in click_ts:
    plt.axvline(t, color="red", alpha=0.8)
plt.xlabel("Time (s)"); plt.ylabel("Amplitude")
plt.title("Audio waveform (from XDF) + mouse-click events")
plt.tight_layout(); plt.show()

# ---- quick sanity check: any clicks after audio stopped? --------------------
extra = click_ts[ click_ts > (aud_ts[-1] - t0) ]
if extra.size:
    print(f"⚠️  {len(extra)} click(s) occur AFTER the last audio sample "
          f"({aud_ts[-1]-t0:.2f}s). "
          "Probably recorded while audio stream had already stopped.")
else:
    print("✓ All clicks lie within the audio recording window.")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\cogpsy-vrlab\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\cogpsy-v

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: initialization failed

In [8]:
#!/usr/bin/env python3
"""
PPS Experiment – XDF Stream Extractor (Dependency-Free Version)

This script loads a LabRecorder .xdf file, finds the Audio and MouseEvents streams,
and saves them as WAV and CSV files without relying on pandas, soundfile, or matplotlib.

Usage:
    python xdf_extractor.py
    
Or modify the paths in the USER CONFIG section below.
"""

import os
import pathlib
import re
import json
import struct
import wave
import csv
from typing import Optional, List, Dict, Any
import xml.etree.ElementTree as ET

# Try to import numpy, with fallback
try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    print("Warning: NumPy not available. Using pure Python fallbacks.")
    HAS_NUMPY = False

# ============================================================================
# SIMPLE XDF LOADER (bypasses pyxdf dependency)
# ============================================================================

def read_xdf_simple(filepath: str) -> tuple:
    """
    Simplified XDF reader that extracts basic stream information.
    Returns (streams, header) where streams is a list of stream dictionaries.
    """
    streams = []
    
    with open(filepath, 'rb') as f:
        # Read XDF header
        header = {}
        
        while True:
            # Read chunk header
            try:
                length_bytes = f.read(4)
                if len(length_bytes) < 4:
                    break
                length = struct.unpack('<I', length_bytes)[0]
                
                if length == 0:
                    continue
                    
                tag = struct.unpack('<H', f.read(2))[0]
                payload = f.read(length - 2)
                
                if tag == 2:  # Stream header
                    stream_info = parse_stream_header(payload)
                    stream = {
                        'info': stream_info,
                        'time_series': [],
                        'time_stamps': []
                    }
                    streams.append(stream)
                    
                elif tag == 3:  # Samples
                    if streams:
                        parse_samples(payload, streams[-1])
                        
                elif tag == 6:  # Stream footer
                    continue
                    
            except struct.error:
                break
                
    return streams, header

def parse_stream_header(payload: bytes) -> dict:
    """Parse stream header XML."""
    try:
        xml_str = payload.decode('utf-8')
        root = ET.fromstring(xml_str)
        
        info = {}
        for elem in root:
            if len(list(elem)) == 0:  # Leaf node
                info[elem.tag] = [elem.text or '']
            else:  # Has children
                info[elem.tag] = {}
                for child in elem:
                    info[elem.tag][child.tag] = [child.text or '']
                    
        return info
    except:
        return {'name': ['Unknown'], 'type': ['Unknown'], 'channel_count': ['1']}

def parse_samples(payload: bytes, stream: dict) -> None:
    """Parse sample data from payload."""
    try:
        channel_count = int(stream['info'].get('channel_count', ['1'])[0])
        sample_format = stream['info'].get('channel_format', ['float32'])[0]
        
        # Simple parsing - assumes float32 format
        if sample_format == 'float32':
            sample_size = 4
        else:
            sample_size = 4  # Default
            
        # Each sample has timestamp (8 bytes) + channel data
        timestamp_size = 8
        record_size = timestamp_size + (channel_count * sample_size)
        
        offset = 0
        while offset + record_size <= len(payload):
            # Read timestamp
            timestamp = struct.unpack('<d', payload[offset:offset+8])[0]
            offset += 8
            
            # Read channel data
            sample_data = []
            for _ in range(channel_count):
                if sample_format == 'float32':
                    value = struct.unpack('<f', payload[offset:offset+4])[0]
                else:
                    value = 0.0
                sample_data.append(value)
                offset += 4
                
            stream['time_stamps'].append(timestamp)
            stream['time_series'].append(sample_data)
            
    except:
        pass  # Skip malformed samples

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def find_stream(streams: List[Dict], *, type_=None, name_regex=None, channel_count=None) -> Optional[Dict]:
    """Return the first stream that matches all given criteria."""
    for s in streams:
        info = s['info']
        name = info.get('name', [''])[0]
        stype = info.get('type', [''])[0]
        nch = int(info.get('channel_count', ['1'])[0])
        
        if type_ is not None and stype != type_:
            continue
        if channel_count is not None and nch != channel_count:
            continue
        if name_regex is not None and re.fullmatch(name_regex, name) is None:
            continue
        return s
    return None

def save_wav_simple(filename: str, data: List[List[float]], sample_rate: int) -> None:
    """Save audio data as WAV file without soundfile dependency."""
    if HAS_NUMPY:
        samples = np.array(data, dtype=np.float32)
        if samples.ndim == 1:
            samples = samples.reshape(-1, 1)
    else:
        # Pure Python version
        samples = data
        if isinstance(data[0], (int, float)):  # Mono
            samples = [[x] for x in data]
    
    n_frames = len(samples)
    n_channels = len(samples[0]) if samples else 1
    
    with wave.open(filename, 'wb') as wav_file:
        wav_file.setnchannels(n_channels)
        wav_file.setsampwidth(2)  # 16-bit
        wav_file.setframerate(sample_rate)
        
        # Convert float32 to int16
        for frame in samples:
            for sample in frame:
                # Clamp to [-1, 1] and convert to 16-bit int
                clamped = max(-1.0, min(1.0, sample))
                int_sample = int(clamped * 32767)
                wav_file.writeframes(struct.pack('<h', int_sample))

def save_csv_simple(filename: str, data: List[List[float]], headers: List[str]) -> None:
    """Save data as CSV file without pandas dependency."""
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(headers)
        for row in data:
            writer.writerow(row)

# ============================================================================
# MAIN EXTRACTION LOGIC
# ============================================================================

def extract_xdf_streams(xdf_path: str, out_dir: str) -> None:
    """Main function to extract streams from XDF file."""
    out_dir = pathlib.Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Loading {xdf_path} ...")
    
    # Try pyxdf first, fall back to simple loader
    streams = None
    try:
        import pyxdf
        streams, _ = pyxdf.load_xdf(xdf_path)
        print(f"Loaded {len(streams)} streams using pyxdf")
    except ImportError:
        print("pyxdf not available, using simple XDF loader...")
        streams, _ = read_xdf_simple(xdf_path)
        print(f"Loaded {len(streams)} streams using simple loader")
    except Exception as e:
        print(f"pyxdf failed: {e}")
        print("Falling back to simple XDF loader...")
        streams, _ = read_xdf_simple(xdf_path)
        print(f"Loaded {len(streams)} streams using simple loader")
    
    if not streams:
        print("No streams found in XDF file!")
        return
    
    # Identify audio stream (2 channels, type 'Audio')
    audio_stream = (
        find_stream(streams, type_="Audio", channel_count=2)
        or find_stream(streams, name_regex=r".*_Audio_.*")
    )
    
    # Identify mouse-click stream (3 channels, type 'MouseEvents')
    mouse_stream = (
        find_stream(streams, type_="MouseEvents", channel_count=3)
        or find_stream(streams, name_regex=r".*_MouseClicks_.*")
    )
    
    print("\n--- streams detected ---")
    print("Audio:", audio_stream['info'].get('name', ['not found'])[0] if audio_stream else "not found")
    print("Mouse:", mouse_stream['info'].get('name', ['not found'])[0] if mouse_stream else "not found")
    
    # Save audio stream
    if audio_stream:
        name = audio_stream["info"]["name"][0]
        sr_str = audio_stream["info"].get("nominal_srate", ["44100"])[0]
        sr = int(float(sr_str))  # Handle "44100.0000" format
        
        ts = audio_stream["time_series"]
        if len(ts) == 0:
            print(f"⚠️  Stream '{name}' contains 0 samples – nothing to write.")
        else:
            wav_path = out_dir / f"{name}.wav"
            save_wav_simple(str(wav_path), ts, sr)
            print(f"✓ Saved audio -> {wav_path}   samples={len(ts)}, sr={sr}")
    else:
        print("No audio stream found")
    
    # Save mouse clicks
    if mouse_stream:
        name = mouse_stream['info']['name'][0]
        click_data = mouse_stream['time_series']
        
        if click_data:
            csv_path = out_dir / f"{name}.csv"
            headers = ['rel_time_sec', 'x_px', 'y_px']
            save_csv_simple(str(csv_path), click_data, headers)
            print(f"✓ Saved mouse clicks -> {csv_path}  rows={len(click_data)}")
        else:
            print("Mouse stream contains no data")
    else:
        print("No mouse stream found")

# ============================================================================
# SIMPLE VISUALIZATION (optional, without matplotlib)
# ============================================================================

def create_simple_plot(xdf_path: str) -> None:
    """Create a simple text-based visualization of the data."""
    print("\n--- Simple Data Analysis ---")
    
    try:
        import pyxdf
        streams, _ = pyxdf.load_xdf(xdf_path)
    except:
        streams, _ = read_xdf_simple(xdf_path)
    
    audio = find_stream(streams, type_="Audio")
    mouse = find_stream(streams, type_="MouseEvents")
    
    if audio and len(audio["time_series"]) > 0:
        if HAS_NUMPY:
            aud_data = np.array(audio["time_series"])
            aud_ts = np.array(audio["time_stamps"])
            duration = aud_ts[-1] - aud_ts[0] if len(aud_ts) > 1 else 0
        else:
            duration = (audio["time_stamps"][-1] - audio["time_stamps"][0] 
                       if len(audio["time_stamps"]) > 1 else 0)
        
        print(f"Audio: {len(audio['time_series'])} samples, {duration:.2f}s duration")
    
    if mouse and len(mouse["time_series"]) > 0:
        if HAS_NUMPY:
            click_ts = np.array(mouse["time_stamps"])
            t0 = audio["time_stamps"][0] if audio and audio["time_stamps"] else 0
            rel_clicks = click_ts - t0
            print(f"Mouse: {len(mouse['time_series'])} clicks")
            print(f"Click times (rel): {rel_clicks[:5]}..." if len(rel_clicks) > 5 else f"Click times (rel): {rel_clicks}")
        else:
            print(f"Mouse: {len(mouse['time_series'])} clicks")

# ============================================================================
# USER CONFIGURATION & MAIN
# ============================================================================

if __name__ == "__main__":
    # ---- USER INPUTS -------------------------------------------------------
    xdf_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\Results\LSL_Output\P001_20250506_171104_R001.xdf"
    out_dir = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\Results\LSL_Output"
    # -----------------------------------------------------------------------
    
    # Check if file exists
    if not os.path.exists(xdf_path):
        print(f"Error: XDF file not found: {xdf_path}")
        print("Please update the xdf_path variable in the script.")
        exit(1)
    
    # Extract streams
    extract_xdf_streams(xdf_path, out_dir)
    
    # Optional: create simple analysis
    create_simple_plot(xdf_path)
    
    print("\n✓ Processing complete!")

Loading C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\Results\LSL_Output\P001_20250506_171104_R001.xdf ...
Loaded 2 streams using pyxdf

--- streams detected ---
Audio: Audio_P001_20250506_171059
Mouse: MouseClicks_P001_20250506_171059


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [ ]:
#!/usr/bin/env python3
"""
PPS Experiment – XDF Stream Extractor (Dependency-Free Version)

This script loads a LabRecorder .xdf file, finds the Audio and MouseEvents streams,
and saves them as WAV and CSV files without relying on pandas, soundfile, or matplotlib.

Usage:
    python xdf_extractor.py
    
Or modify the paths in the USER CONFIG section below.
"""

import os
import pathlib
import re
import json
import struct
import wave
import csv
from typing import Optional, List, Dict, Any
import xml.etree.ElementTree as ET

# Try to import numpy, with fallback
try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    print("Warning: NumPy not available. Using pure Python fallbacks.")
    HAS_NUMPY = False

# ============================================================================
# SIMPLE XDF LOADER (bypasses pyxdf dependency)
# ============================================================================

def read_xdf_simple(filepath: str) -> tuple:
    """
    Simplified XDF reader that extracts basic stream information.
    Returns (streams, header) where streams is a list of stream dictionaries.
    """
    streams = []
    
    with open(filepath, 'rb') as f:
        # Read XDF header
        header = {}
        
        while True:
            # Read chunk header
            try:
                length_bytes = f.read(4)
                if len(length_bytes) < 4:
                    break
                length = struct.unpack('<I', length_bytes)[0]
                
                if length == 0:
                    continue
                    
                tag = struct.unpack('<H', f.read(2))[0]
                payload = f.read(length - 2)
                
                if tag == 2:  # Stream header
                    stream_info = parse_stream_header(payload)
                    stream = {
                        'info': stream_info,
                        'time_series': [],
                        'time_stamps': []
                    }
                    streams.append(stream)
                    
                elif tag == 3:  # Samples
                    if streams:
                        parse_samples(payload, streams[-1])
                        
                elif tag == 6:  # Stream footer
                    continue
                    
            except struct.error:
                break
                
    return streams, header

def parse_stream_header(payload: bytes) -> dict:
    """Parse stream header XML."""
    try:
        xml_str = payload.decode('utf-8')
        root = ET.fromstring(xml_str)
        
        info = {}
        for elem in root:
            if len(list(elem)) == 0:  # Leaf node
                info[elem.tag] = [elem.text or '']
            else:  # Has children
                info[elem.tag] = {}
                for child in elem:
                    info[elem.tag][child.tag] = [child.text or '']
                    
        return info
    except:
        return {'name': ['Unknown'], 'type': ['Unknown'], 'channel_count': ['1']}

def parse_samples(payload: bytes, stream: dict) -> None:
    """Parse sample data from payload."""
    try:
        channel_count = int(stream['info'].get('channel_count', ['1'])[0])
        sample_format = stream['info'].get('channel_format', ['float32'])[0]
        
        # Simple parsing - assumes float32 format
        if sample_format == 'float32':
            sample_size = 4
        else:
            sample_size = 4  # Default
            
        # Each sample has timestamp (8 bytes) + channel data
        timestamp_size = 8
        record_size = timestamp_size + (channel_count * sample_size)
        
        offset = 0
        while offset + record_size <= len(payload):
            # Read timestamp
            timestamp = struct.unpack('<d', payload[offset:offset+8])[0]
            offset += 8
            
            # Read channel data
            sample_data = []
            for _ in range(channel_count):
                if sample_format == 'float32':
                    value = struct.unpack('<f', payload[offset:offset+4])[0]
                else:
                    value = 0.0
                sample_data.append(value)
                offset += 4
                
            stream['time_stamps'].append(timestamp)
            stream['time_series'].append(sample_data)
            
    except:
        pass  # Skip malformed samples

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def find_stream(streams: List[Dict], *, type_=None, name_regex=None, channel_count=None) -> Optional[Dict]:
    """Return the first stream that matches all given criteria."""
    for s in streams:
        info = s['info']
        name = info.get('name', [''])[0]
        stype = info.get('type', [''])[0]
        nch = int(info.get('channel_count', ['1'])[0])
        
        if type_ is not None and stype != type_:
            continue
        if channel_count is not None and nch != channel_count:
            continue
        if name_regex is not None and re.fullmatch(name_regex, name) is None:
            continue
        return s
    return None

def save_wav_simple(filename: str, data: List[List[float]], sample_rate: int) -> None:
    """Save audio data as WAV file without soundfile dependency."""
    if HAS_NUMPY:
        samples = np.array(data, dtype=np.float32)
        if samples.ndim == 1:
            samples = samples.reshape(-1, 1)
        n_frames = samples.shape[0]
        n_channels = samples.shape[1]
    else:
        # Pure Python version
        samples = data
        if len(data) > 0 and isinstance(data[0], (int, float)):  # Mono
            samples = [[x] for x in data]
        n_frames = len(samples)
        n_channels = len(samples[0]) if len(samples) > 0 else 1
    
    with wave.open(filename, 'wb') as wav_file:
        wav_file.setnchannels(n_channels)
        wav_file.setsampwidth(2)  # 16-bit
        wav_file.setframerate(sample_rate)
        
        # Convert float32 to int16
        if HAS_NUMPY:
            for i in range(n_frames):
                for j in range(n_channels):
                    # Clamp to [-1, 1] and convert to 16-bit int
                    sample = float(samples[i, j])
                    clamped = max(-1.0, min(1.0, sample))
                    int_sample = int(clamped * 32767)
                    wav_file.writeframes(struct.pack('<h', int_sample))
        else:
            for frame in samples:
                for sample in frame:
                    # Clamp to [-1, 1] and convert to 16-bit int
                    clamped = max(-1.0, min(1.0, sample))
                    int_sample = int(clamped * 32767)
                    wav_file.writeframes(struct.pack('<h', int_sample))

def save_csv_simple(filename: str, data: List[List[float]], headers: List[str]) -> None:
    """Save data as CSV file without pandas dependency."""
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(headers)
        for row in data:
            writer.writerow(row)

# ============================================================================
# MAIN EXTRACTION LOGIC
# ============================================================================

def extract_xdf_streams(xdf_path: str, out_dir: str) -> None:
    """Main function to extract streams from XDF file."""
    out_dir = pathlib.Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Loading {xdf_path} ...")
    
    # Try pyxdf first, fall back to simple loader
    streams = None
    try:
        import pyxdf
        streams, _ = pyxdf.load_xdf(xdf_path)
        print(f"Loaded {len(streams)} streams using pyxdf")
    except ImportError:
        print("pyxdf not available, using simple XDF loader...")
        streams, _ = read_xdf_simple(xdf_path)
        print(f"Loaded {len(streams)} streams using simple loader")
    except Exception as e:
        print(f"pyxdf failed: {e}")
        print("Falling back to simple XDF loader...")
        streams, _ = read_xdf_simple(xdf_path)
        print(f"Loaded {len(streams)} streams using simple loader")
    
    if not streams:
        print("No streams found in XDF file!")
        return
    
    # Identify audio stream (2 channels, type 'Audio')
    audio_stream = (
        find_stream(streams, type_="Audio", channel_count=2)
        or find_stream(streams, name_regex=r".*_Audio_.*")
    )
    
    # Identify mouse-click stream (3 channels, type 'MouseEvents')
    mouse_stream = (
        find_stream(streams, type_="MouseEvents", channel_count=3)
        or find_stream(streams, name_regex=r".*_MouseClicks_.*")
    )
    
    print("\n--- streams detected ---")
    print("Audio:", audio_stream['info'].get('name', ['not found'])[0] if audio_stream else "not found")
    print("Mouse:", mouse_stream['info'].get('name', ['not found'])[0] if mouse_stream else "not found")
    
    # Save audio stream
    if audio_stream:
        name = audio_stream["info"]["name"][0]
        sr_str = audio_stream["info"].get("nominal_srate", ["44100"])[0]
        sr = int(float(sr_str))  # Handle "44100.0000" format
        
        ts = audio_stream["time_series"]
        if len(ts) == 0:
            print(f"⚠️  Stream '{name}' contains 0 samples – nothing to write.")
        else:
            wav_path = out_dir / f"{name}.wav"
            save_wav_simple(str(wav_path), ts, sr)
            print(f"✓ Saved audio -> {wav_path}   samples={len(ts)}, sr={sr}")
    else:
        print("No audio stream found")
    
    # Save mouse clicks
    if mouse_stream:
        name = mouse_stream['info']['name'][0]
        click_data = mouse_stream['time_series']
        
        if click_data:
            csv_path = out_dir / f"{name}.csv"
            headers = ['rel_time_sec', 'x_px', 'y_px']
            save_csv_simple(str(csv_path), click_data, headers)
            print(f"✓ Saved mouse clicks -> {csv_path}  rows={len(click_data)}")
        else:
            print("Mouse stream contains no data")
    else:
        print("No mouse stream found")

# ============================================================================
# SIMPLE VISUALIZATION (optional, without matplotlib)
# ============================================================================

def create_simple_plot(xdf_path: str) -> None:
    """Create a simple text-based visualization of the data."""
    print("\n--- Simple Data Analysis ---")
    
    try:
        import pyxdf
        streams, _ = pyxdf.load_xdf(xdf_path)
    except:
        streams, _ = read_xdf_simple(xdf_path)
    
    audio = find_stream(streams, type_="Audio")
    mouse = find_stream(streams, type_="MouseEvents")
    
    if audio and len(audio["time_series"]) > 0:
        if HAS_NUMPY:
            aud_data = np.array(audio["time_series"])
            aud_ts = np.array(audio["time_stamps"])
            duration = aud_ts[-1] - aud_ts[0] if len(aud_ts) > 1 else 0
        else:
            duration = (audio["time_stamps"][-1] - audio["time_stamps"][0] 
                       if len(audio["time_stamps"]) > 1 else 0)
        
        print(f"Audio: {len(audio['time_series'])} samples, {duration:.2f}s duration")
    
    if mouse and len(mouse["time_series"]) > 0:
        if HAS_NUMPY:
            click_ts = np.array(mouse["time_stamps"])
            t0 = audio["time_stamps"][0] if audio and audio["time_stamps"] else 0
            rel_clicks = click_ts - t0
            print(f"Mouse: {len(mouse['time_series'])} clicks")
            print(f"Click times (rel): {rel_clicks[:5]}..." if len(rel_clicks) > 5 else f"Click times (rel): {rel_clicks}")
        else:
            print(f"Mouse: {len(mouse['time_series'])} clicks")

# ============================================================================
# USER CONFIGURATION & MAIN
# ============================================================================

if __name__ == "__main__":
    # ---- USER INPUTS -------------------------------------------------------
    xdf_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\Results\LSL_Output\P001_20250506_171104_R001.xdf"
    out_dir = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level3_Analysis\Output\PPS_RT"
    # -----------------------------------------------------------------------
    
    # Check if file exists
    if not os.path.exists(xdf_path):
        print(f"Error: XDF file not found: {xdf_path}")
        print("Please update the xdf_path variable in the script.")
        exit(1)
    
    # Extract streams
    extract_xdf_streams(xdf_path, out_dir)
    
    # Optional: create simple analysis
    create_simple_plot(xdf_path)
    
    print("\n✓ Processing complete!")